**This problem set is due Wednesday, October 14, 2026 at 11:59 pm. Please plan ahead and submit your work on time.**

# Problem Set 05: Adversarial Games

In this problem set, you will explore minimax, alpha-beta pruning, and expectimax.

0. [Credit for Contributors (required)](#contributors)

1. [Adversarial Games (85 points)](#problem2)
    1. [MiniMax (35 points)](#part1)
    2. [MiniMax with Alpha-Beta pruning (25 points)](#part2)
    3. [Expectimax (25 points)](#part3)
2. [Exploiting a Near-Optimal Opponent [AI-enabled] (10 points)](#ai_problem)
    1. [Use an LLM to Model a Near-Optimal Opponent [AI-enabled] (5 points)](#part2A)
    2. [Visualize the Opponent's Weakness [AI-enabled] (5 points)](#part2B)
3. [Time Spent on Pset (5 points)](#part4)

    
**100 points** total for Problem Set 5

## Imports and Utilities

In [ ]:
from __future__ import division
import random
from utils import *

%load_ext autoreload
%autoreload 2
from principles_of_autonomy.grader import Grader
from principles_of_autonomy.notebook_tests.pset_5 import TestPSet5

## <a name="contributors"></a> 0. Credit for Contributors

List the various students, lecture notes, or online resouces that helped you complete this problem set:

Ex: I worked with Bob on the cat expectimax problem.

<div class="alert alert-info">
Write your answer in the cell below this one.
</div>

--> *(double click on this cell to delete this text and type your answer here)*

## <a name="problem2"></a> 1. Adversarial Games: The Cut-Throat Game of Tic-Tac-Toe

Though the game of tic-tac-toe is very simple, it serves as a good example for MiniMax search, alpha-beta pruning, and Expectimax search because it is fully solvable. In this problem set, you will implement MiniMax search with and without alpha-beta pruning, and compare performance. You will then simulate a full game to prove that an optimal strategy in tic-tac-toe can only guarantee a tie (against an optimal adversary). Then you will implement Expectimax search to play against a suboptimal adversary.

We have defined for you a `game_state` class, which works for general games, and a `tic_tac_toe_board` class, both defined in the file `utils.py`. You may want to look in this file now to see the specifics of these classes.

A `tic_tac_toe_board` object is defined as follows:

```
tic_tac_toe_board(symbols)
```

The `symbols` argument is a list of 9 elements which may be one of `'x'`, `'o'`, or `' '`, corresponding to the symbols on the board supplied row by row.

You will not work directly with a `tic_tac_toe_board`. Instead, a `game_state` object will track the board state of any two player game object supplied to it. It will also provide information on the current player, and the state of the game in the last move. An instance is defined as:

```
game_state(board,parent=None)
```

The `board` argument is any game board (e.g., a tic-tac-toe board), and the `parent` argument refers to the previous game state. If `None` is supplied as the parent, the game is assumed to have started from this state.

<a name="board_state"></a>The current board of a `game_state` is stored in `state.board.moves`: the same list of 9 symbols, read row by row, so the cells are numbered

```
0 | 1 | 2
--+---+--
3 | 4 | 5
--+---+--
6 | 7 | 8
```

For example, `state.board.moves[4]` is the symbol in the center cell. Despite its name, `state.board.moves` is a snapshot of the current position, not a history of the moves played. Each successor is a new `game_state` holding a copy of its parent's board with one more symbol added, so a `game_state` only records how it was reached through its chain of `parent` states.

To show the current board, you may use `print(game_state)` as below.

In [ ]:
# Define a game state for tic-tac-toe
example_state = game_state(tic_tac_toe_board(['x','o','x','x','x','o','o',' ',' ']))
# View the state of the board
print(example_state)

By default, a `game_state` is initialized assuming player 1, who plays x's and attempts to maximize score, is playing first. Player 2 plays o's and attempts to minimize the score. 

We attribute a score of +1 to any game state where x (player 1) wins, -1 to any game state where o (player 2) wins, and 0 to any tied or incomplete game state. You may see the score using `game_state.score()`, and check whether a game is complete using `game_state.terminal_check()`.

In [ ]:
# Run this cell to see some example games, who has won, and whether the game is complete.
example_state = game_state(tic_tac_toe_board(['x','o','x','x','x','o','o',' ',' ']))
print(example_state)
print('Game complete? {}'.format(example_state.terminal_check()))
print('Score of this game is {}: Game incomplete.\n'.format(example_state.score()))

example_state = game_state(tic_tac_toe_board(['x','o','x','x','x','o','o',' ','x']))
print(example_state)
print('Game complete? {}'.format(example_state.terminal_check()))
print('Score of this game is {}: x wins.\n'.format(example_state.score()))

example_state = game_state(tic_tac_toe_board(['x','o','x','x','x','o','o','o','o']))
print(example_state)
print('Game complete? {}'.format(example_state.terminal_check()))
print('Score of this game is {}: o wins.\n'.format(example_state.score()))

example_state = game_state(tic_tac_toe_board(['x','o','x','x','x','o','o','x','o']))
print(example_state)
print('Game complete? {}'.format(example_state.terminal_check()))
print('Score of this game is {}: Game tie.\n'.format(example_state.score()))

The player who needs to play next in any given `game_state` is given through the command `game_state.player`. The function `game_state.successors()` creates a list of `game_state` objects that could be reachable within a move of the current player. The parent of the successors is automatically allocated.

In [ ]:
# Run this cell to see current players and successor states
example_state = game_state(tic_tac_toe_board(['x','o','x','x',' ','o','o',' ',' ']))
print('Example state:')
print(example_state)
print('Next player is {} and the possible successor states are:\n'.format(example_state.player))
successor_states = example_state.successors()
for s in successor_states:
    print(s)
    
print('-----------------------------------------\n')
example_state = successor_states[0]
print('Example state:')
print(example_state)
print('Next player is {} and the possible successor states are:\n'.format(example_state.player))
successor_states = example_state.successors()
for s in successor_states:
    print(s)

### 1A. <a name="part1"/> </a>Implement MiniMax Search (35 points)

We will use MiniMax to produce a game where both players are following their optimal strategy. You should be able to convince yourself that in this case, the outcome of the entire game can be defined at the start. 

You will define the following functions:

```python
def maximize_score(state: game_state) -> (max_score, states_explored, optimal_play)
```
```python
def minimize_score(state: game_state) -> (min_score, states_explored, optimal_play)
```

Both functions take a `game_state` object as input. It is assumed that when `maximize_score` is run, the next player to play is player 1, while `minimize_score` is run when the next player is player 2.  

The output of `maximize_score` will be a tuple `(max_score, states_explored, optimal_play)`, where: 

- `max_score` is the maximum score that can be achieved starting from the provided state if both players follow an optimum strategy. 
- `states_explored` is the total number of game states that have been considered to perform this computation (including the current state). If your search reaches the same state multiple times, count it multiple times.
- `optimal_play` is a list of `game_state` objects following the optimal game `[state, state2, state3, ..., final_state]` from the current state (leftmost element of list) to its conclusion (rightmost element).

Similarly, the output of `minimize_score` will be a tuple `(min_score, states_explored, optimal_play)`, where:

- `min_score` is the minimum score that can be achieved starting from the provided state if both players follow an optimum strategy. 
- `states_explored` and `optimal_play` are defined as above.

**In the case where multiple moves give a tie for the optimal play, the `optimal_play` list should follow the first successor in the list generated by `state.successors()`.**

We provide you with the base case function `score(state)` (or `value(state)` in lecture). Given a game state, this function runs a terminal check and if the game state is NOT terminal, it calls the minimizer or maximier node (as determined by `state.player`).

<div class="alert alert-info">
Implement functions `maximize_score(state)` and `minimize_score(state)` below.
</div>

In [ ]:
# Write your code for MiniMax search here.

# This is the base case function.
def score(state):
    # Initialize states_explored as 1 (this state) and optimal_play (this state).
    states_explored = 1
    optimal_play = [state]
    
    if state.terminal_check():
        terminal_score = state.score()
        return (terminal_score, states_explored, optimal_play)
    elif state.player == 1:
        return maximize_score(state)
    elif state.player == 2:
        return minimize_score(state)
    
def maximize_score(state):
    # Check this is player 1, the maximizer.
    if state.player != 1:
        raise Error
    
    raise NotImplementedError()


def minimize_score(state):
    # Check this is player 2, the minimizer.
    if state.player != 2:
        raise Error
    
    raise NotImplementedError()

In [ ]:
"""Test your MiniMax search code here."""
Grader.run_single_test_inline(TestPSet5, "test_1_minimax", locals())

### 1B. <a name="part2"/> </a>Implement Alpha-Beta Pruning (25 points)

Now you will implement alpha-beta pruning in your MiniMax functions and demonstrate that the same result is achieved with fewer game states needing to be searched. 

You will define the following functions:

```python
def maximize_score_alpha_beta(state,alpha,beta)
def minimize_score_alpha_beta(state,alpha,beta)
```

These functions will look very similar to your implementations of `maximize_score` and `minimize_score` (in fact, a good place to start is by copying and pasting your code from above), except they will take the alpha and beta parameters defined by alpha-beta pruning. The output format should be the same as the corresponding functions without pruning. Once again, we provide you with the base case function `score_alpha_beta`.

<div class="alert alert-info">
Implement functions `maximize_score_alpha_beta(state,alpha,beta)` and `minimize_score_alpha_beta(state,alpha,beta)` below.
</div>

In [ ]:
# Write your code for MiniMax search with alpha-beta pruning here.

# This is the base case function.
def score_alpha_beta(state,alpha,beta):
    # Initialize states_explored as 1 (this state) and optimal_play (this state).
    states_explored = 1
    optimal_play = [state]
    
    if state.terminal_check():
        terminal_score = state.score()
        return (terminal_score, states_explored, optimal_play)
    elif state.player == 1:
        return maximize_score_alpha_beta(state,alpha,beta)
    elif state.player == 2:
        return minimize_score_alpha_beta(state,alpha,beta)

def maximize_score_alpha_beta(state,alpha,beta):
    # Check this is player 1, the maximizer.
    if state.player != 1:
        raise Error
    
    raise NotImplementedError()

def minimize_score_alpha_beta(state,alpha,beta):
    # Check this is player 2, the minimizer.
    if state.player != 2:
        raise Error
        
    raise NotImplementedError()

In [ ]:
"""Test your alpha beta search code here."""
Grader.run_single_test_inline(TestPSet5, "test_2_alpha_beta", locals())

### Simulate a Full Game w/ Optimal Agent

You're now prepared to run a full game of tic-tac-toe from an initially blank state. If it was possible to always guarantee a win against a perfect opponent, you would see x's win the game. Unfortunately, this isn't possible! Let's see this by running your code both with and without alpha-beta pruning.

In [ ]:
# Run this cell to see an optimum game of tic-tac-toe. Note the performance WITHOUT alpha-beta pruning.
# This may take a few seconds. 
blank_game = game_state(tic_tac_toe_board([' ',' ',' ',' ',' ',' ',' ',' ',' ']))
(max_score, states_explored, optimal_play) = score(blank_game)
print('Max score = {}'.format(max_score))
print('States explored = {}'.format(states_explored))
for s in optimal_play:
    print(s)

In [ ]:
# Run this cell to see an optimum game of tic-tac-toe. Note the performance gain WITH alpha-beta pruning.  
blank_game = game_state(tic_tac_toe_board([' ',' ',' ',' ',' ',' ',' ',' ',' ']))
(max_score, states_explored, optimal_play) = score_alpha_beta(blank_game,-2,2)
print('Max score = {}'.format(max_score))
print('States explored = {}'.format(states_explored))
for s in optimal_play:
    print(s)

### 1C. <a name="part3"/> </a>Implement Expectimax Search (25 points)

MiniMax and alpha-beta are great, but they both assume that you are playing against an adversary who makes optimal decisions. As anyone who has ever won tic-tac-toe can tell you, this is not always the case. In this question you will implement Expectimax Search, which is useful for modeling probabilistic behavior of agents who may make suboptimal choices. For this version of the problem, you will play against a random adversary.

You will define the following functions:

```python
def maximize_score_expectimax(state)
def expected_score_expectimax(state)
```

Once again, `maximize_score_expectimax` will look very similar to your implementation of `maximize_score`. However, `expected_score_expectimax`'s `optimal_play` should choose amongst successors at random. Assume equal probability of choosing amongst successors. The output format should be the same as the corresponding MiniMax functions.

<div class="alert alert-info">
Implement functions `maximize_score_expectimax(state)` and `expected_score_expectimax(state)` below.
</div>

In [ ]:
# Write your code for expectimax search here.

# This is the base case function.
def score_expectimax(state):
    # Initialize states_explored as 1 (this state) and optimal_play (this state).
    states_explored = 1
    optimal_play = [state]
    
    if state.terminal_check():
        terminal_score = state.score()
        return (terminal_score, states_explored, optimal_play)
    elif state.player == 1:
        return maximize_score_expectimax(state)
    elif state.player == 2:
        return expected_score_expectimax(state)

def maximize_score_expectimax(state):
    # Check this is player 1, the maximizer.
    if state.player != 1:
        raise Error
    
    raise NotImplementedError()


def expected_score_expectimax(state):
    # Check this is player 2, the minimizer.
    if state.player != 2:
        raise Error
    
    raise NotImplementedError()

In [ ]:
"""Test your expectimax search code here."""
Grader.run_single_test_inline(TestPSet5, "test_3_expectimax", locals())

### Simulate a Full Game w/ Suboptimal Agent

You're now prepared to run a full game of tic-tac-toe from an initially blank state, this time with an agent that isn't necessarily optimal.

In [ ]:
# Run this cell to see a suboptimal game of tic-tac-toe. Run it multiple times. Notice how we always win. 
blank_game = game_state(tic_tac_toe_board([' ',' ',' ',' ',' ',' ',' ',' ',' ']))
(max_score, states_explored, optimal_play) = score_expectimax(blank_game)
print('Max score = {}'.format(max_score))
print('States explored = {}'.format(states_explored))
for s in optimal_play:
    print(s)

## <a name="ai_problem"></a> 2. Exploiting a Near-Optimal Opponent [AI-enabled] (10 points)

In Problem 1C, Expectimax assumed the opponent chooses uniformly at random. In this problem you will use a Large Language Model to build a better model of an opponent from a record of its games, use that model to beat it, and then explain its weakness with a visualization.

### 2A. <a name="part2A"/> </a>Use an LLM to Model a Near-Optimal Opponent [AI-enabled] (5 points)

Use an LLM (ChatGPT, Claude, etc.) to come up with the probabilities to model a _near_-optimal opponent for ExpectiMax search. Your agent will be playing _second_ (e.g. O's) against the opponent (who is playing X's), and its objective is to exploit your opponent's suboptimality to win more than 20 out of 100 repeated games.  We will provide an initial hint at the opponent's strategy below, but your main evidence is `opponent_games.txt`: a record of 200 games between the opponent and a player who chose randomly among its non-losing moves. Use an LLM to help you parse this record and find where, and roughly how often, the opponent deviates from optimal play, then turn what you find into your model.

Implement the function `opponent_model(state)`, which, given a `game_state` where X (the opponent) is about to move, returns the probability that X chooses each successor in `state.successors()` (same order, non-negative, summing to 1). The autograder will play O using an autograder-implementation of Expectimax with your model. Note: the grader will pass `opponent_model` a `state` from the board alone, without the moves that led to it: `state.parent` is `None` and `state.player` is 1 (x to move). Use `state.board.moves` to see the position (see [how the board state is stored](#board_state)).

Hint: your opponent will play perfectly using a min-max strategy, except in one particular scenario will make a mistake with some probability $p$. Can you identify that mistake and estimate that probability? You may find it useful to use your minimax implementation to determine the perfect play strategy when computing your mistake probabilities.

<div class="alert alert-info">
Implement functions `opponent_model(game_state)` below.
</div>

In [ ]:
def opponent_model(state) -> list[float]:
    """Probability that X (the opponent) picks each successor in state.successors().

    state: a game_state in which X is about to move.
    Returns a list of probabilities in the same order as state.successors().
    """
    successors = state.successors()

    

    # Placeholder model: x picks uniformly at random. Replace this with your own!
    return [1 / len(successors)] * len(successors)

In [ ]:
"""Play 100 games against the hidden opponent using your model, and print every game."""
Grader.run_single_test_inline(TestPSet5, "test_4_opponent_model", locals())

### 2B. <a name="part2B"/> </a>Visualize the Opponent's Weakness [AI-enabled] (5 points)


<div class="alert alert-info">
In the space below, with the help of an LLM, generate a visualization that shows the weakness in your opponent's strategy. You may use either printed text (e.g., boards printed with <code>print(state)</code>) or <code>matplotlib</code>. In particular, the visualization must show where the opponent deviates from optimal play, and why that deviation lets O win.
</div>

Some suggestions for visualization ideas:
- showing, in the position(s) where the opponent goes wrong, how often it plays each move in `opponent_games.txt`, and which of those moves MiniMax considers optimal,
- following a game forward from the opponent's mistake to show how O wins from there.

In [ ]:
### Visualization Code here.


For fun: a neat [visualization](https://xkcd.com/832/) of the recursive game tree.

## <a name="part4"></a> Time Spent on Pset (5 points)

Please use [this form](https://forms.gle/ev8waEiJBN97HxV68) to tell us how long you spent on this pset. After you submit the form, the form will give you a confirmation word. Please enter that confirmation word below to get 5 points. 

In [ ]:
form_confirmation_word = "ENTER_THE_CONFIRMATION_WORD_HERE"

In [ ]:
# Run all tests. The autograder total is 95 points; Section 2B (5 points) is graded by hand and is not included.
Grader.grade_output([TestPSet5], [locals()], "results.json")
Grader.print_test_results("results.json")